## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-24-00-48-00 +0000,wsj,"Nikkei Rises 2.7%, Led by Chip, Metals Stocks",https://www.wsj.com/finance/stocks/nikkei-rise...
1,2026-11-24-00-46-46 +0000,chitri,Arlington Heights gateway developer’s next bui...,https://www.chicagotribune.com/2026/11/23/arli...
2,2026-11-24-00-45-53 +0000,bbc,Chris Mason: Can Burnham turn volume down on E...,https://www.bbc.co.uk/news/articles/c6pw47jyyz...
3,2026-11-24-00-43-03 +0000,nyt,Turning the Lens Around: Bob Woodward Takes On...,https://www.nytimes.com/2026/11/23/us/politics...
4,2026-11-24-00-42-18 +0000,nyt,Virginia School District Shaken by Sex Abuse C...,https://www.nytimes.com/2026/11/23/us/albemarl...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2592/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
42,huang,54
39,nvidia,47
41,jensen,41
84,trump,41
43,ceo,23


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
45,2026-11-23-23-10-27 +0000,bbc,Nvidia reinstates CEO Jensen Huang after affai...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,243
24,2026-11-23-23-46-05 +0000,nypost,Five wild Trump quotes about reinstated Nvidia...,https://nypost.com/2026/11/23/us-news/five-wil...,223
223,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,221
220,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...,220
149,2026-11-23-19-46-00 +0000,wsj,When President Trump wanted Jensen Huang back ...,https://www.wsj.com/politics/policy/nvidia-ceo...,218


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
45,243,2026-11-23-23-10-27 +0000,bbc,Nvidia reinstates CEO Jensen Huang after affai...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...
69,86,2026-11-23-22-29-00 +0000,wsj,"Rep. Jimmy Gomez (D., Calif.) said he would re...",https://www.wsj.com/politics/democratic-lawmak...
331,67,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...
323,57,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
12,45,2026-11-24-00-11-21 +0000,nypost,California families break down as toxic explos...,https://nypost.com/2026/11/23/us-news/chiquita...
100,43,2026-11-23-21-45-00 +0000,wsj,‘I Fell Off My Chair.’ Investors Marvel as Hua...,https://www.wsj.com/finance/stocks/i-fell-off-...
353,37,2026-11-23-04-55-00 +0000,wsj,Asian Regulators Step Up Efforts to Calm AI-Dr...,https://www.wsj.com/finance/stocks/asian-regul...
30,36,2026-11-23-23-35-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
133,36,2026-11-23-20-30-16 +0000,nypost,Russia has lost astonishing 700K soldiers sinc...,https://nypost.com/2026/11/23/world-news/russi...
8,33,2026-11-24-00-31-28 +0000,bbc,No 10 North will face greater scrutiny as Burn...,https://www.bbc.co.uk/news/articles/c6rrg7jx5g...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
